# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

from pytorch_tabular.models import CategoryEmbeddingModelConfig, TabNetModelConfig

from pathlib import Path
import sys

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    StatsModelsEstimator,
    PytorchTabularEstimator,
    update_estimator_params,
    OptunaOptimizer,
    GridSearchOptimizer,
    RANDOM_STATE,
)

# from pytorch_lightning.loggers import MLFlowLogger
from pytorch_lightning.loggers import TensorBoardLogger

seed = RANDOM_STATE

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Settings

In [4]:
model_postfix = 'first_launch'

scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
step_scoring_average = "mean"
# n_trials = 1500 # checked with 15 and 100

save_model_and_metrics = True
metrics_file = "metrics_modelling4_6-neural_networks.xlsx"

## MultiLayer Perceptron (MLP)

In [5]:
estimator = PytorchTabularEstimator
target = "splashing"

add_smote = True
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}

estimator_params = {
    "model_class": CategoryEmbeddingModelConfig,
    "model_config_params": {
        'layers': '128-64-32',
        'activation': 'LeakyReLU',
        # 'activation_config': {'negative_slope': 0.01},
        'dropout': 0.2,
        'batch_norm_continuous_input': True, # We have normalized features
        'learning_rate': 1e-3,
    },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 200,
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 10,
        # 'output_dir': Path("..", "results", "models_modelling4"), # take from ml_pipe
        # 'log_target': 'wandb',
        # 'logger_name': 'default',
        # 'checkpoints': 'f1_macro',
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

ml_pipe = MLPipeline(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
    metrics_file=metrics_file,
)

ml_pipe.run(
    save_model_and_metrics=save_model_and_metrics,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

2025-04-25 18:00:44,755 - {pytorch_tabular.tabular_model:146} - INFO - Experiment Tracking is turned off

Seed set to 42


...

2025-04-25 18:02:47,410 - {pytorch_tabular.tabular_model:689} - INFO - Training the model completed

2025-04-25 18:02:47,412 - {pytorch_tabular.tabular_model:1529} - INFO - Loading the best model

no summary in estimator class "PytorchTabularEstimator"


,0
target,splashing
model,CategoryEmbeddingModel_splashing_smote_first_l...
holdout_test_f1_macro,0.823391
holdout_test_accuracy_balanced,0.818287
holdout_test_roc_auc,0.909722
holdout_test_f1,0.877551
holdout_test_accuracy,0.84
cv_test_f1_macro_median,0.842262
cv_test_accuracy_balanced_median,0.859133
cv_test_roc_auc_median,0.941176


Model saved in ../results/models_modelling4/CategoryEmbeddingModel_splashing_smote_first_launch


In [6]:
estimator = PytorchTabularEstimator
target = "no_fragmentation"

add_smote = True
is_smotenc = True
smote_params = {
    'categorical_features': (
        'wettability',
        'inclination',
    ),
}

estimator_params = {
    "model_class": CategoryEmbeddingModelConfig,
    "model_config_params": {
        'activation': 'LeakyReLU',
        'dropout': 0.2,
        'batch_norm_continuous_input': True, # We have normalized features
        'learning_rate': 1e-3,
    },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 200,
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 10,
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

ml_pipe = MLPipeline(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
    metrics_file=metrics_file,
)

ml_pipe.run(
    save_model_and_metrics=save_model_and_metrics,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

2025-04-25 18:02:48,071 - {pytorch_tabular.tabular_model:146} - INFO - Experiment Tracking is turned off

Seed set to 42


...

2025-04-25 18:04:07,897 - {pytorch_tabular.tabular_model:689} - INFO - Training the model completed

2025-04-25 18:04:07,898 - {pytorch_tabular.tabular_model:1529} - INFO - Loading the best model

no summary in estimator class "PytorchTabularEstimator"


,0
target,no_fragmentation
model,CategoryEmbeddingModel_no_fragmentation_smoten...
holdout_test_f1_macro,0.853293
holdout_test_accuracy_balanced,0.870455
holdout_test_roc_auc,0.97
holdout_test_f1,0.790698
holdout_test_accuracy,0.88
cv_test_f1_macro_median,0.875762
cv_test_accuracy_balanced_median,0.902564
cv_test_roc_auc_median,0.965812


Model saved in ../results/models_modelling4/CategoryEmbeddingModel_no_fragmentation_smotenc_first_launch


## TabNet

In [7]:
estimator = PytorchTabularEstimator
target = "splashing"

add_smote = True
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}

estimator_params = {
    "model_class": TabNetModelConfig,
    "model_config_params": {
        'n_d': 8, # decoder dimension
        'n_a': 8, # attention dimension
        'n_steps': 3, # number of steps of the attention mechanism
        'gamma': 1.2, # sparsity parameter
        'n_independent': 1, # number of independent GLU layers (shared across decision steps)
        'n_shared': 1, # number of shared GLU layers (shared across steps & features)
        'virtual_batch_size': 32, # batch size for Ghost Batch Normalization
        'learning_rate': 1e-3,
    },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 300,
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 15,
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

ml_pipe = MLPipeline(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
    metrics_file=metrics_file,
)

ml_pipe.run(
    save_model_and_metrics=save_model_and_metrics,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

2025-04-25 18:04:08,202 - {pytorch_tabular.tabular_model:146} - INFO - Experiment Tracking is turned off

Seed set to 42


...

2025-04-25 18:12:11,002 - {pytorch_tabular.tabular_model:689} - INFO - Training the model completed

2025-04-25 18:12:11,006 - {pytorch_tabular.tabular_model:1529} - INFO - Loading the best model

no summary in estimator class "PytorchTabularEstimator"


,0
target,splashing
model,TabNetModel_splashing_smote_first_launch
holdout_test_f1_macro,0.75
holdout_test_accuracy_balanced,0.763889
holdout_test_roc_auc,0.865741
holdout_test_f1,0.8
holdout_test_accuracy,0.76
cv_test_f1_macro_median,0.766862
cv_test_accuracy_balanced_median,0.7887
cv_test_roc_auc_median,0.852941


Model saved in ../results/models_modelling4/TabNetModel_splashing_smote_first_launch


In [8]:
estimator = PytorchTabularEstimator
target = "no_fragmentation"

add_smote = True
is_smotenc = True
smote_params = {
    'categorical_features': (
        'wettability',
        'inclination',
    ),
}

estimator_params = {
    "model_class": TabNetModelConfig,
    "model_config_params": {
        'n_d': 8, # decoder dimension
        'n_a': 8, # attention dimension
        'n_steps': 3, # number of steps of the attention mechanism
        'gamma': 1.2, # sparsity parameter
        'n_independent': 1, # number of independent GLU layers (shared across decision steps)
        'n_shared': 1, # number of shared GLU layers (shared across steps & features)
        'virtual_batch_size': 32, # batch size for Ghost Batch Normalization
        'learning_rate': 1e-3,
    },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 300,
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 15,
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

ml_pipe = MLPipeline(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
    metrics_file=metrics_file,
)

ml_pipe.run(
    save_model_and_metrics=save_model_and_metrics,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

2025-04-25 18:12:12,506 - {pytorch_tabular.tabular_model:146} - INFO - Experiment Tracking is turned off

Seed set to 42


...

2025-04-25 18:20:38,638 - {pytorch_tabular.tabular_model:689} - INFO - Training the model completed

2025-04-25 18:20:38,642 - {pytorch_tabular.tabular_model:1529} - INFO - Loading the best model

no summary in estimator class "PytorchTabularEstimator"


,0
target,no_fragmentation
model,TabNetModel_no_fragmentation_smotenc_first_launch
holdout_test_f1_macro,0.874582
holdout_test_accuracy_balanced,0.911364
holdout_test_roc_auc,0.96
holdout_test_f1,0.826087
holdout_test_accuracy,0.893333
cv_test_f1_macro_median,0.820946
cv_test_accuracy_balanced_median,0.858974
cv_test_roc_auc_median,0.919414


Model saved in ../results/models_modelling4/TabNetModel_no_fragmentation_smotenc_first_launch
